# 야생동물 전투 시뮬레이션

### 0. API키 로드

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

### 1. 야생 동물 객체 생성

In [2]:
import random

def select_enemy():
    animals = ['들개', '원숭이', '곰']
    feature = [
    '온순함(공격 의사 낮음)',
    '경계심이 강함(가까이 가면 공격할 수도 있음)',
    '허기짐(먹이를 탐색 중임)',
    '매우 흉포함'
    ]

    enemy = random.choice(animals)
    enemy_feature = random.choice(feature)

    current_state = {
        'user_state': f'특이사항 없음. {enemy} 발견',
        'enemy_state': f'{enemy_feature} 상태로 유저 발견',
        'history_summary': '전투가 막 시작했습니다.'
    }

    print(f'상대할 야생 동물: {enemy}\n상태: {enemy_feature}')

    return enemy, enemy_feature, current_state

In [3]:
enemy, enemy_feature, turn_0 = select_enemy()
print(enemy)
print(enemy_feature)
print(turn_0)

상대할 야생 동물: 곰
상태: 매우 흉포함
곰
매우 흉포함
{'user_state': '특이사항 없음. 곰 발견', 'enemy_state': '매우 흉포함 상태로 유저 발견', 'history_summary': '전투가 막 시작했습니다.'}


### 2. 심판 생성

##### 2-1. Prompt 생성

In [4]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate

def generate_prompt():
    system_template = '''
    [Role] 
    당신은 인간 vs 짐승 전투 시뮬레이션의 심판입니다. 상대는 {enemy}입니다. 매우 현실적이고 논리적으로 결과를 판정하세요.

    [현재 턴] {turn_count}

    [지금까지의 전투 요약]
    {battle_history}

    [현재 상태]
    - 유저 상태: {user_status} (현재 HP: {user_hp})
    - {enemy} 상태: {enemy_state} (현재 HP: {enemy_hp})

    위 상황에 대한 이번 턴 유저 행동입니다.
    [이번 턴 유저 행동]
    {user_input_text}

    이 행동의 성공 여부와 결과를 판정하고, 다음 턴에 입력으로 들어갈 내용을 정밀하게 갱신하세요.
    '''

    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(system_template)
    ])

    return prompt

##### 2-2. Output 형태

In [5]:
from pydantic import BaseModel, Field

def generate_output_form():
    class NextTurnState(BaseModel):
        user_state: str = Field(description='다음 턴을 위한 유저의 신체 및 상황 요약 (예: 우측 팔 골절, 백초크 유지 중 등)')
        enemy_state: str = Field(description='다음 턴을 위한 동물의 신체 및 상황 요약 (예: 호흡 곤란, 목에 상처 등)')
        history_summary: str = Field(description='처음부터 지금까지의 누적 전투 상황을 1~2줄로 요약 (맥락 유지 용도)')

    class TurnResult(BaseModel):
        referee_decision: str = Field(description='이번 턴 유저 행동에 대한 결과 및 판정 묘사')
        enemy_action: str = Field(description='상대 동물의 반응 및 다음 행동 묘사')
        turn_summary: str = Field(description='이번 턴 전투의 짧은 요약 (예: 유저는 곰에게 다가가 백초크를 시도했지만, 곰이 목을 휘감은 오른팔을 할퀴며 초크가 풀어짐.)')
        user_damage: int = Field(description='유저가 입은 피해량 (0 ~ 100 사이의 정수, 0은 피해 없음, 100은 즉사)')
        enemy_damage: int = Field(description='상대 동물이 입은 피해량 (0 ~ 100 사이의 정수, 0은 피해 없음, 100은 즉사)')
        next_turn_state: NextTurnState

    return NextTurnState, TurnResult

##### 2-3. LLM 모델 생성 및 Chain 생성

In [6]:
from langchain.chat_models import init_chat_model

def generate_chain(prompt, TurnResult):
    load_dotenv()
    llm = init_chat_model('openai:gpt-4.1-mini')
    llm = llm.with_structured_output(TurnResult)
    chain = prompt | llm
    return chain

### 3. 룰 설명 및 객체 설명

In [7]:
def rule(enemy, enemy_feature):
    desc = f'''"그정도는 내가 이기지~"
    평소 "야, 원숭이 정도는 싸우면 내가 이기지!"라고 허세를 부리셨나요?
    좋습니다. 지금부터 당신의 그 자신감이 생물학적 한계를 넘어서 승리를 쟁취할 수 있을지 증명해 보세요!

    [전투 규칙]
    1. 턴제 음성 전투: 당신의 행동을 마이크에 대고 자세하게 말씀해 주세요.
    2. 냉철한 AI 심판: AI심판은 과학적, 해부학적 사실만을 바탄으로 당신의 행동을 평가합니다.
    3. HP 시스템: 당신과 야생 동물의 기본 HP는 100입니다. 먼저 HP가 0이 되는 쪽이 패배합니다.
    4. 창의적인 행동: 야생 동물과 맨몸으로 싸우는 것은 자살 행위입니다. 주변 환경과 도구를 창의적으로 활용하면 생존 확률이 증가합니다.

    [오늘의 상대]
    야생 동물 종: {enemy}
    현재 상태: {enemy_feature}

    준비가 되셨다면, 아래의 [시작] 버튼을 눌러 당신이 '최강의 동물'임을 증명하십시오.
    '''
    return print(desc)

### 4. 전투 방식 입력 (STT)

In [29]:
import speech_recognition as sr
import threading, time

def get_voice_input(time_limit=5):
    recognizer = sr.Recognizer()

    with sr.Microphone() as source:
        print('다음 행동을 말씀해 주세요.')
        recognizer.adjust_for_ambient_noise(source, duration=0.5)
        audio = recognizer.record(source, duration=time_limit)

        user_input_text = recognizer.recognize_google(audio, language='ko-KR')
    
    return user_input_text

In [30]:
user_input_text = get_voice_input()
print(user_input_text)

다음 행동을 말씀해 주세요.
들깨 옆으로 천천히 다가가 길로틴 초크


### 5. Chain 호출

In [ ]:
def play_turn(chain, user_input_text: str, enemy: str, current_state: dict, user_hp: int, enemy_hp: int, turn_count: int):
    response = chain.invoke({
        "enemy": enemy,
        "turn_count": turn_count,
        "battle_history": current_state["history_summary"],
        "user_status": current_state["user_state"],
        "user_hp": user_hp,
        "enemy_state": current_state["enemy_state"],
        "enemy_hp": enemy_hp,
        "user_input_text": user_input_text
    })
    
    user_hp = max(0, user_hp - response.user_damage)
    enemy_hp = max(0, enemy_hp - response.enemy_damage)
    
    current_state = {
        "user_state": response.next_turn_state.user_state,
        "enemy_state": response.next_turn_state.enemy_state,
        "history_summary": response.next_turn_state.history_summary
    }
    
    turn_count += 1

    return response, user_hp, enemy_hp, current_state, turn_count

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display
import io

def play_voice(text):
    if not text:
        return
        
    try:
        tts = gTTS(text=text, lang='ko')
        filename = "temp_voice.mp3"
        tts.save(filename)

        display(Audio(filename, autoplay=True))
        
    except Exception as e:
        print(f"음성 재생 중 오류 발생: {e}")

In [27]:
play_voice('안녕하세요')
play_voice('곰은 피를 흘리며 쓰러졌습니다. 당신이 승리하였습니다.')
play_voice('들개가 당신의 공격을 피하고 오른쪽 팔을 물었습니다. 당신은 오른쪽 팔에 심각한 부상을 입었습니다.')

In [ ]:
enemy, enemy_feature, current_state = select_enemy()

rule(enemy, enemy_feature)

prompt = generate_prompt()
NextTurnState, TurnResult = generate_output_form()
chain = generate_chain(prompt, TurnResult)

user_input_text = ''
user_hp = 100
enemy_hp = 100
turn_count = 1

while True:
    print(f'[{turn_count}턴]\n나의 HP: {user_hp} | {enemy}의 HP: {enemy_hp}]')
    user_input_text = input('행동을 입력해주세요.(종료: q)')
    if user_input_text == 'q':
        print('당신은 도망쳤습니다.')
        break

    print('===== 결과를 기다리는 중 =====')
    response, user_hp, enemy_hp, current_state, turn_count = play_turn(chain, user_input_text, enemy, current_state, user_hp, enemy_hp, turn_count)
    print(f'[당신의 행동]\n{user_input_text}\n\n')
    print(f'[심판의 판정]\n{response.referee_decision}\n\n')
    print(f'[{enemy}의 행동]\n{response.enemy_action}\n\n')
    print(f'[이번 턴 전투 요약]\n{response.turn_summary}\n\n')
    print(f'[피해량]\n나: {response.user_damage}\n{enemy}: {response.enemy_damage}')

    if user_hp == 0:
        print('당신은 사망했습니다.')
        break
    elif enemy_hp == 0:
        print(f'{enemy}를 쓰러뜨렸습니다!')
        break

상대할 야생 동물: 들개
상태: 온순함(공격 의사 낮음)
"그정도는 내가 이기지~"
    평소 "야, 원숭이 정도는 싸우면 내가 이기지!"라고 허세를 부리셨나요?
    좋습니다. 지금부터 당신의 그 자신감이 생물학적 한계를 넘어서 승리를 쟁취할 수 있을지 증명해 보세요!

    [전투 규칙]
    1. 턴제 음성 전투: 당신의 행동을 마이크에 대고 자세하게 말씀해 주세요.
    2. 냉철한 AI 심판: AI심판은 과학적, 해부학적 사실만을 바탄으로 당신의 행동을 평가합니다.
    3. HP 시스템: 당신과 야생 동물의 기본 HP는 100입니다. 먼저 HP가 0이 되는 쪽이 패배합니다.
    4. 창의적인 행동: 야생 동물과 맨몸으로 싸우는 것은 자살 행위입니다. 주변 환경과 도구를 창의적으로 활용하면 생존 확률이 증가합니다.

    [오늘의 상대]
    야생 동물 종: 들개
    현재 상태: 온순함(공격 의사 낮음)

    준비가 되셨다면, 아래의 [시작] 버튼을 눌러 당신이 '최강의 동물'임을 증명하십시오.
    
[1턴]
나의 HP: 100 | 들개의 HP: 100]
===== 결과를 기다리는 중 =====
[당신의 행동]
공격을 하지 않는 척 천천히 들개의 오른쪽으로 다가가다가 나와 평행해졌을 때 재빠르게 길로틴 초크를 건다


[심판의 판정]
유저가 들개의 주의를 끌지 않기 위해 천천히 접근했고, 평행할 때 재빠르게 길로틴 초크를 시도했다. 들개가 온순한 상태이나 본능적으로 위협을 느껴 저항했으나, 유저의 신속한 움직임과 정확한 자세 덕분에 초크가 거의 완벽하게 들어갔다. 들개는 호흡 곤란 상태에 빠지며 큰 피해를 입었다.


[들개의 행동]
들개는 갑작스러운 공격에 당황했으나 즉시 저항하며 초크에서 벗어나려 애썼다.


[이번 턴 전투 요약]
유저가 들개의 오른쪽으로 천천히 접근해 평행해진 뒤 신속하게 길로틴 초크를 성공시켰다. 들개는 저항했으나 초크로 인해 심한 호흡 곤란에 빠졌다.


[피해량]
나: 0
들개: 40
[2턴]


5. 전투 결과 출력 (TTS)